In [1]:

import pandas as pd

# Download Customer Churn Dataset
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(url)

# Display first 5 rows
print("Dataset downloaded successfully!")

print("\nDataset Shape:")
print(df.shape)

print("\nFirst 5 Rows:")
display(df.head())

print("\nColumn Names:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

# Save dataset as CSV
df.to_csv("customer_churn.csv", index=False)

print("\nCSV file saved successfully!")


Dataset downloaded successfully!

Dataset Shape:
(7043, 21)

First 5 Rows:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes



Column Names:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Missing Values:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

CSV file saved successfully!


In [2]:

# STEP 2: DATA PREPROCESSING AND MODEL TRAINING

import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Load dataset
df = pd.read_csv("customer_churn.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

# 2. Clean TotalCharges column
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

# Remove customer ID
df.drop(columns=["customerID"], inplace=True)

# 3. Separate input features and target
X = df.drop(columns=["Churn"])
y = df["Churn"].map({"Yes": 1, "No": 0})

# 4. Identify numerical and categorical features
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("\nNumerical Features:", numerical_features)
print("\nCategorical Features:", categorical_features)

# 5. Numerical preprocessing
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# 6. Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# 7. Combine preprocessing
preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

# 8. Create complete ML pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# 9. Split dataset into training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 10. Train model
model.fit(X_train, y_train)

print("\nModel training completed!")

# 11. Evaluate model
predictions = model.predict(X_test)

print("\nModel Accuracy:",
      accuracy_score(y_test, predictions))

print("\nClassification Report:")
print(classification_report(y_test, predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

# 12. Save trained pipeline
with open("model.pkl", "wb") as file:
    pickle.dump(model, file)

print("\nSUCCESS: model.pkl saved!")


Dataset loaded successfully!
Dataset shape: (7043, 21)

Numerical Features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical Features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Model training completed!

Model Accuracy: 0.8055358410220014

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409


Confusion Matrix:
[[926 109]
 [165 209]]

SUCCESS: model.pkl saved!


In [4]:

# STEP 3: STREAMLIT FRONTEND APPLICATION

!pip -q install streamlit

app_code = '''
import streamlit as st
import pandas as pd
import pickle

st.set_page_config(
    page_title="Customer Churn Prediction",
    page_icon="📊",
    layout="wide"
)

# Load trained model
@st.cache_resource
def load_model():
    with open("model.pkl", "rb") as file:
        return pickle.load(file)

model = load_model()

# Application title
st.title("📊 Customer Churn Prediction")
st.write(
    "Enter customer details to predict whether "
    "the customer is likely to leave the company."
)

st.divider()

# Input form
st.header("Customer Information")

with st.form("prediction_form"):

    col1, col2 = st.columns(2)

    with col1:
        gender = st.selectbox("Gender", ["Male", "Female"])
        senior = st.selectbox("Senior Citizen", [0, 1])
        partner = st.selectbox("Partner", ["Yes", "No"])
        dependents = st.selectbox("Dependents", ["Yes", "No"])
        tenure = st.slider("Tenure (Months)", 0, 72, 12)
        phone = st.selectbox("Phone Service", ["Yes", "No"])
        multiple = st.selectbox(
            "Multiple Lines",
            ["Yes", "No", "No phone service"]
        )
        internet = st.selectbox(
            "Internet Service",
            ["DSL", "Fiber optic", "No"]
        )
        security = st.selectbox(
            "Online Security",
            ["Yes", "No", "No internet service"]
        )
        backup = st.selectbox(
            "Online Backup",
            ["Yes", "No", "No internet service"]
        )

    with col2:
        protection = st.selectbox(
            "Device Protection",
            ["Yes", "No", "No internet service"]
        )
        support = st.selectbox(
            "Tech Support",
            ["Yes", "No", "No internet service"]
        )
        tv = st.selectbox(
            "Streaming TV",
            ["Yes", "No", "No internet service"]
        )
        movies = st.selectbox(
            "Streaming Movies",
            ["Yes", "No", "No internet service"]
        )
        contract = st.selectbox(
            "Contract",
            ["Month-to-month", "One year", "Two year"]
        )
        paperless = st.selectbox(
            "Paperless Billing", ["Yes", "No"]
        )
        payment = st.selectbox(
            "Payment Method",
            [
                "Electronic check",
                "Mailed check",
                "Bank transfer (automatic)",
                "Credit card (automatic)"
            ]
        )
        monthly = st.number_input(
            "Monthly Charges",
            min_value=0.0,
            value=70.0
        )
        total = st.number_input(
            "Total Charges",
            min_value=0.0,
            value=840.0
        )

    submitted = st.form_submit_button(
        "Predict Customer Churn",
        use_container_width=True
    )

# Prediction
if submitted:

    input_data = pd.DataFrame([{
        "gender": gender,
        "SeniorCitizen": senior,
        "Partner": partner,
        "Dependents": dependents,
        "tenure": tenure,
        "PhoneService": phone,
        "MultipleLines": multiple,
        "InternetService": internet,
        "OnlineSecurity": security,
        "OnlineBackup": backup,
        "DeviceProtection": protection,
        "TechSupport": support,
        "StreamingTV": tv,
        "StreamingMovies": movies,
        "Contract": contract,
        "PaperlessBilling": paperless,
        "PaymentMethod": payment,
        "MonthlyCharges": monthly,
        "TotalCharges": total
    }])

    prediction = model.predict(input_data)[0]
    probability = model.predict_proba(input_data)[0][1]

    st.divider()
    st.header("Prediction Result")

    if prediction == 1:
        st.error("⚠️ Customer is likely to churn!")
    else:
        st.success("✅ Customer is likely to stay!")

    st.metric(
        "Churn Probability",
        f"{probability:.2%}"
    )

    st.progress(float(probability))
'''

# Save Streamlit application
with open("app.py", "w") as file:
    file.write(app_code)

print("SUCCESS! app.py created and Streamlit frontend configured.")


SUCCESS! app.py created and Streamlit frontend configured.


In [5]:

requirements = """
pandas
numpy
scikit-learn
streamlit
"""

with open("requirements.txt", "w") as file:
    file.write(requirements)

print("requirements.txt created successfully!")


requirements.txt created successfully!


In [6]:
import os

print("Project files:")

for file in ["customer_churn.csv", "model.pkl", "app.py", "requirements.txt"]:
    print(file, ":", "Available" if os.path.exists(file) else "Missing")

Project files:
customer_churn.csv : Available
model.pkl : Available
app.py : Available
requirements.txt : Available


In [7]:

import os
import shutil
import zipfile
from pathlib import Path

# Create project folder
project = Path("customer-churn-prediction")
project.mkdir(exist_ok=True)

# Create data folder
data_folder = project / "data"
data_folder.mkdir(exist_ok=True)

# Copy dataset
shutil.copy("customer_churn.csv", data_folder / "customer_churn.csv")

# Copy trained model and application
shutil.copy("model.pkl", project / "model.pkl")
shutil.copy("app.py", project / "app.py")
shutil.copy("requirements.txt", project / "requirements.txt")

# Create README
readme = """
# Customer Churn Prediction

## Problem Statement
Predict whether a telecom customer will leave the company.

## Dataset
IBM Telco Customer Churn Dataset.

## Technologies
- Python
- Pandas
- NumPy
- Scikit-learn
- Streamlit

## Machine Learning Model
Logistic Regression.

## Preprocessing
- Missing value imputation
- StandardScaler
- OneHotEncoder
- ColumnTransformer
- Scikit-learn Pipeline

## Run Instructions

Install dependencies:
pip install -r requirements.txt

Run the application:
streamlit run app.py

## Deployment
Streamlit Community Cloud.

## Author
Your Name
"""

(project / "README.md").write_text(readme)

# Create .gitignore
(project / ".gitignore").write_text(
    "__pycache__/\n.venv/\n*.pyc\n"
)

# Check Python syntax
for filename in ["app.py"]:
    compile((project / filename).read_text(), filename, "exec")

# Create ZIP
zip_name = "customer-churn-prediction.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for file in project.rglob("*"):
        if file.is_file():
            z.write(file, file.as_posix())

print("PROJECT CREATED SUCCESSFULLY!")
print("\nProject files:")

for file in project.rglob("*"):
    if file.is_file():
        print(file)

print("\nZIP FILE:", zip_name)

# Download ZIP
from google.colab import files
files.download(zip_name)


PROJECT CREATED SUCCESSFULLY!

Project files:
customer-churn-prediction/requirements.txt
customer-churn-prediction/model.pkl
customer-churn-prediction/README.md
customer-churn-prediction/.gitignore
customer-churn-prediction/app.py
customer-churn-prediction/data/customer_churn.csv

ZIP FILE: customer-churn-prediction.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>